# Sentence-count-only chunk analysis

Uses the PDF extraction, cleaning, and page-by-page spaCy sentence rules from `ingest_new.py`. Chunks contain consecutive **whole sentences**, with no character limit, token-based splitting, or truncation. Token lengths are measured afterward using BGE's own tokenizer, including special tokens.

The recommendation is the **largest tested sentence count with zero chunks exceeding 512 tokens** for this PDF. This is a length-safety recommendation, not a retrieval-quality optimum. No inference API requests or Supabase writes are made.

In [ ]:
# Run once in your selected notebook kernel if dependencies are missing:
# %pip install -r requirements-notebook.txt

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

PROJECT = Path.cwd()
if not (PROJECT / "ingest_new.py").exists():
    raise FileNotFoundError("Open this notebook from the project folder containing ingest_new.py.")
sys.path.insert(0, str(PROJECT))
import ingest_new as ingest

OUTPUT = PROJECT / "chunk_analysis_results"
OUTPUT.mkdir(exist_ok=True)
CANDIDATES = list(range(1, 21))
MODEL = ingest.EMBEDDING_MODEL
assert MODEL == "BAAI/bge-small-en-v1.5", "Recheck context and query rules after changing models."
print("PDF:", ingest.PDF_PATH.name)
print("Model:", MODEL)
print("PDF SHA256:", hashlib.sha256(ingest.PDF_PATH.read_bytes()).hexdigest())

## 1. Import and preprocess the PDF

The imported functions remove recurring headers/footers and standalone page numbers, normalize Unicode, repair line-break hyphenation, join layout newlines and collapse whitespace. Sentence splitting remains page-by-page, exactly as in the script.

In [ ]:
pages = ingest.extract_pdf_pages(ingest.PDF_PATH)
cleaned_pages = ingest.clean_pdf_pages(pages)
nlp = ingest.build_sentence_splitter()
sentences = [
    {"text": sent.text.strip(), "page_number": page["page_number"]}
    for page in cleaned_pages
    for sent in nlp(page["text"]).sents
    if sent.text.strip()
]
assert sentences, "No machine-readable sentences were extracted."
display(pd.DataFrame([{
    "pdf_pages": len(pages), "cleaned_pages": len(cleaned_pages),
    "sentences": len(sentences),
    "cleaned_characters": sum(len(p["text"]) for p in cleaned_pages),
}]))
display(pd.DataFrame(sentences[:5]))

## 2. Load the embedding model's tokenizer

The encoder's output dimension (384) is unrelated to chunk length. Its **input context is 512 tokens**. Token counts include [CLS]/[SEP]. We disable padding and truncation so overlong chunks are visible instead of silently shortened.

In [ ]:
tokenizer_path = hf_hub_download(MODEL, "tokenizer.json", token=False)
revision = Path(tokenizer_path).parent.name
config_path = hf_hub_download(MODEL, "sentence_bert_config.json", revision=revision, token=False)
model_config = json.loads(Path(config_path).read_text(encoding="utf-8"))
TOKEN_LIMIT = int(model_config["max_seq_length"])
assert TOKEN_LIMIT == 512, model_config
tokenizer = Tokenizer.from_file(tokenizer_path)
tokenizer.no_truncation()
tokenizer.no_padding()
print("Pinned tokenizer revision:", revision)
print("Context limit:", TOKEN_LIMIT)
print("Special tokens for an empty input:", len(tokenizer.encode("").ids))

def token_lengths(texts):
    result = []
    for start in range(0, len(texts), 256):
        result.extend(len(e.ids) for e in tokenizer.encode_batch(texts[start:start + 256]))
    return np.asarray(result, dtype=np.int64)

def describe_chunks(texts, n, experiment):
    lengths = token_lengths(texts)
    return {
        "experiment": experiment, "sentences_setting": n, "chunks": len(texts),
        "mean_tokens": float(lengths.mean()),
        "std_tokens": float(lengths.std(ddof=0)),
        "median_tokens": float(np.median(lengths)),
        "p95_tokens": float(np.percentile(lengths, 95)),
        "max_tokens": int(lengths.max()),
        "over_limit_chunks": int((lengths > TOKEN_LIMIT).sum()),
        "over_limit_pct": float(100 * (lengths > TOKEN_LIMIT).mean()),
        "mean_context_pct": float(100 * lengths.mean() / TOKEN_LIMIT),
    }, lengths

# Standard deviation is population SD: we measure the entire PDF's chunks.
# The final, possibly shorter chunk is included for every candidate.

## 3. Compare sentence counts 1-20

Call the actual ingestion chunker for each setting. Only `SENTENCES_PER_CHUNK` changes. Verify the exact sentence counts, retained text, and final remainder chunk. Token measurements never change chunk boundaries. Population standard deviation includes every chunk, including the last one.

In [ ]:
rows, distributions = [], {}
original_setting = ingest.SENTENCES_PER_CHUNK
try:
    for n in CANDIDATES:
        ingest.SENTENCES_PER_CHUNK = n
        chunks = ingest.create_sentence_chunks(cleaned_pages)
        texts = [c["content"] for c in chunks]
        expected_texts = [" ".join(s["text"] for s in sentences[start:start+n])
                          for start in range(0, len(sentences), n)]
        assert texts == expected_texts, "Chunker changed sentence content or boundaries."
        assert all(c["metadata"]["sentence_count"] == n for c in chunks[:-1])
        assert sum(c["metadata"]["sentence_count"] for c in chunks) == len(sentences)
        stats, lengths = describe_chunks(texts, n, "whole sentences only")
        rows.append(stats)
        distributions[n] = lengths
finally:
    ingest.SENTENCES_PER_CHUNK = original_setting
results = pd.DataFrame(rows)
display(results.drop(columns="experiment").round(2))
results.to_csv(OUTPUT / "chunk_statistics.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].errorbar(results.sentences_setting, results.mean_tokens, yerr=results.std_tokens,
                marker="o", capsize=3, label="Mean +/- population SD")
axes[0].plot(results.sentences_setting, results.p95_tokens, "--", label="95th percentile")
axes[0].axhline(TOKEN_LIMIT, color="crimson", linestyle=":", label="512-token limit")
axes[0].set(title="Whole-sentence chunk lengths", xlabel="Sentences per chunk",
            ylabel="Input tokens, including special tokens", ylim=(0, None))
axes[0].legend(fontsize=8)
axes[1].bar(results.sentences_setting, results.over_limit_pct, color="#d6604d")
axes[1].set(title="Chunks exceeding model context", xlabel="Sentences per chunk",
            ylabel="Chunks over 512 tokens (%)")
for ax in axes:
    ax.set_xticks([1, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
    ax.grid(axis="y", alpha=.2)
fig.savefig(OUTPUT / "chunk_lengths.png", dpi=180)
plt.show()
